In [17]:
import pandas as pd
import os
os.chdir("/home/stavz/masters/gdc/APM/")
from pipeline.config import PATHS, PRIMARY_GENES

In [4]:
mirs = pd.read_csv("data/miRNA/mirtar.csv")

In [9]:
PATHS.targetscan_predictions

PosixPath('/home/stavz/masters/gdc/APM/data/miRNA/Predicted_Targets_Context_Scores.default_predictions.txt')

In [46]:
mirs["Experiments"].value_counts()

Experiments
CLIP-seq                                                                                                                                                                  2756392
PAR-CLIP                                                                                                                                                                   625972
HITS-CLIP                                                                                                                                                                  560951
CLASH                                                                                                                                                                       17310
Microarray                                                                                                                                                                  10884
                                                                                                  

In [42]:
mirs["Species (miRNA)"].value_counts()

Species (miRNA)
hsa     4004479
ebv         395
kshv         37
mmu          18
vun          10
ptr          10
hcmv          5
rno           5
gga           3
ssc           2
sja           2
chi           1
Name: count, dtype: int64

In [21]:
primary_genes_set = set(PRIMARY_GENES)

In [22]:
mirs_genes_set = set(mirs["Target Gene"].unique().tolist())


In [24]:
print(f"Number of primary genes: {len(primary_genes_set)}")
print(f"Number of miRNA targets: {len(mirs_genes_set)}")

# Find genes in primary_genes_set that are not in mirs_genes_set
missing_genes = primary_genes_set - mirs_genes_set

print(f"Number of missing genes: {len(missing_genes)}")
print(missing_genes)


Number of primary genes: 66
Number of miRNA targets: 16981
Number of missing genes: 3
{'RAET1G', 'PSMB10', 'PSME2'}


In [29]:
"MECL1" in mirs_genes_set

False

In [31]:
mirs["Target Gene (Entrez ID)"] = mirs["Target Gene (Entrez ID)"].astype(int)

,miRTarBase ID,miRNA,Species (miRNA),Target Gene,Target Gene (Entrez ID),Species (Target Gene),Experiments,Support Type,References (PMID)
105401,MIRT670374,hsa-miR-514a-5p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23313552.0
105402,MIRT670373,hsa-miR-383-3p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23313552.0
105403,MIRT629520,hsa-miR-6499-5p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23313552.0
105404,MIRT670372,hsa-miR-326,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23313552.0
105405,MIRT670371,hsa-miR-330-5p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23313552.0
...,...,...,...,...,...,...,...,...,...
925048,MIRT629519,hsa-miR-4743-5p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23824327.0
925049,MIRT629518,hsa-miR-4284,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23824327.0
925050,MIRT629517,hsa-miR-24-3p,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23824327.0
925051,MIRT629516,hsa-miR-3178,hsa,ULBP3,79465,hsa,HITS-CLIP,Functional MTI (Weak),23824327.0


In [79]:
import re
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd


# =============================================================================
# LOAD
# =============================================================================

# load mirtarbase data
mirs = pd.read_csv("data/miRNA/mirtar.csv")

# load miR family data (targetscan families)
families = pd.read_csv(
    "data/miRNA/miR_Family_Info.txt",
    sep="\t"
)





# =============================================================================
# CONFIG
# =============================================================================

# Normalized support types we will use everywhere
SUPPORT_TYPES = [
    "Functional MTI",
    "Functional MTI (Weak)",
    "Non-Functional MTI",
    "Non-Functional MTI (Weak)",
]

# Experiment classes you asked to stratify by
EXPERIMENT_CLASSES = [
    "reporter",
    "binding",
    "protein",
    "rna",
    "perturbation",
    "other",
]


# =============================================================================
# COLUMN DETECTION
# =============================================================================

def find_first_existing_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def infer_mirtarbase_columns(df: pd.DataFrame):
    """
    Try to infer the important miRTarBase columns from common header variants.
    Adjust/add candidates here if your file uses different names.
    """
    colmap = {
        "mirtarbase_id": find_first_existing_column(df, [
            "miRTarBase ID", "miRTarBase_ID", "MIRTarBase ID", "MIRTarBase_ID"
        ]),
        "mirna": find_first_existing_column(df, [
            "miRNA", "miRNA Name", "Mature miRNA", "miRNA_name"
        ]),
        "mirna_species": find_first_existing_column(df, [
            "Species (miRNA)", "miRNA Species", "Species_miRNA"
        ]),
        "gene": find_first_existing_column(df, [
            "Target Gene", "Target gene", "Gene", "Target Symbol", "Target"
        ]),
        "gene_entrez": find_first_existing_column(df, [
            "Target Gene (Entrez ID)", "Entrez ID", "Target Entrez ID"
        ]),
        "gene_species": find_first_existing_column(df, [
            "Species (Target Gene)", "Target Gene Species", "Species_Target"
        ]),
        "experiments": find_first_existing_column(df, [
            "Experiments", "Experiment", "Experimental Methods"
        ]),
        "support_type": find_first_existing_column(df, [
            "Support Type", "SupportType"
        ]),
        "references": find_first_existing_column(df, [
            "References", "Reference", "PMID", "PMIDs", "PubMed ID", "Source"
        ]),
    }

    missing = [k for k, v in colmap.items() if k in ["mirna", "gene", "experiments", "support_type"] and v is None]
    if missing:
        raise ValueError(
            f"Could not infer required columns: {missing}\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return colmap


COLS = infer_mirtarbase_columns(mirs)
print("Detected columns:")
for k, v in COLS.items():
    print(f"  {k}: {v}")


# =============================================================================
# NORMALIZATION HELPERS
# =============================================================================

def normalize_support_type(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).strip()

    # normalize spacing/casing variants
    x_low = x.lower().replace("_", " ").replace("  ", " ")
    x_low = re.sub(r"\s+", " ", x_low)

    if x_low == "functional mti":
        return "Functional MTI"
    if x_low == "functional mti (weak)":
        return "Functional MTI (Weak)"
    if x_low == "non-functional mti":
        return "Non-Functional MTI"
    if x_low == "non-functional mti (weak)":
        return "Non-Functional MTI (Weak)"

    # fallback: keep original if new variant appears
    return x


def normalize_gene_symbol(x: str) -> str:
    if pd.isna(x):
        return np.nan
    return str(x).strip()


def normalize_mirna_name(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", "", x)
    return x


def simple_mirna_family(mirna: str) -> str:
    """
    Simple family proxy:
      hsa-miR-122-5p -> miR-122
      hsa-let-7a-5p  -> let-7a
      mmu-miR-21-3p  -> miR-21

    This is NOT seed-family exactness like TargetScan families,
    but it is a good first-pass family label.
    """
    if pd.isna(mirna):
        return np.nan

    m = str(mirna).strip()

    # remove species prefix like hsa-, mmu-, rno-
    m = re.sub(r"^[A-Za-z]{3}-", "", m)

    # remove arm suffix -5p / -3p
    m = re.sub(r"-(5p|3p)$", "", m, flags=re.IGNORECASE)

    return m


def split_experiments(exp_str: str):
    """
    miRTarBase often stores experiments as strings like:
      'Luciferase reporter assay//Western blot//qRT-PCR'

    Return a cleaned list.
    """
    if pd.isna(exp_str):
        return []

    s = str(exp_str).strip()

    # Common separators
    parts = re.split(r"//|;|,|\|", s)
    parts = [p.strip() for p in parts if p.strip()]
    return parts


def classify_experiment(exp: str) -> str:
    """
    Collapse detailed experiments into broad classes.
    Edit rules here if you want different ontology.
    """
    e = str(exp).strip().lower()

    reporter_kw = [
        "luciferase", "reporter assay", "reporter", "gfp reporter"
    ]
    binding_kw = [
        "clip", "hits-clip", "par-clip", "clash", "rip", "rip-chip",
        "pull-down", "pulldown", "immunoprecipitation", "binding assay"
    ]
    protein_kw = [
        "western blot", "elisa", "flow cytometry", "immunoblot", "protein assay",
        "immunohistochemistry", "ihc"
    ]
    rna_kw = [
        "qrt-pcr", "qpcr", "rt-pcr", "northern blot", "microarray",
        "rna-seq", "sequencing", "expression profiling", "pcr"
    ]
    perturbation_kw = [
        "overexpression", "knockdown", "transfection", "sirna", "shrna",
        "mimic", "inhibitor", "antisense", "gene silencing"
    ]

    if any(k in e for k in reporter_kw):
        return "reporter"
    if any(k in e for k in binding_kw):
        return "binding"
    if any(k in e for k in protein_kw):
        return "protein"
    if any(k in e for k in rna_kw):
        return "rna"
    if any(k in e for k in perturbation_kw):
        return "perturbation"
    return "other"


def extract_pmids(ref_value):
    """
    Extract PMIDs from a references/source column.
    If no PMID-like digits are found, return a fallback pseudo-ID based on raw text.
    This preserves study-level collapsing as much as possible.
    """
    if pd.isna(ref_value):
        return []

    s = str(ref_value)

    # collect number strings of length >= 5 as likely PMIDs
    pmids = re.findall(r"\b\d{5,9}\b", s)
    pmids = list(dict.fromkeys(pmids))  # deduplicate preserving order

    if pmids:
        return pmids

    # fallback: use the whole string as a pseudo-study ID
    s = s.strip()
    if not s:
        return []
    return [f"REF::{s}"]


def dict_to_json(d):
    return json.dumps(d, ensure_ascii=False, sort_keys=True)


# =============================================================================
# PREP / NORMALIZE RAW TABLE
# =============================================================================

families = families[families["Species ID"] == 9606].copy()

families = families.rename(columns={
    "miR family": "mir_family",
    "MiRBase ID": "mirna"
})

families["mirna_norm"] = families["mirna"].map(normalize_mirna_name)
mirna_to_family = (
    families
    .drop_duplicates("mirna_norm")
    .set_index("mirna_norm")["mir_family"]
    .to_dict()
)

df = mirs.copy()
df["miRNA_norm"] = df[COLS["mirna"]].map(normalize_mirna_name)
df["gene_norm"] = df[COLS["gene"]].map(normalize_gene_symbol)

# =============================================================================
# FILTER TO PRIMARY GENES
# =============================================================================

if PRIMARY_GENES is not None:
    before = len(df)

    df = df[df["gene_norm"].isin(PRIMARY_GENES)].copy()

    after = len(df)

    print(f"Filtered to PRIMARY_GENES:")
    print(f"  rows before: {before:,}")
    print(f"  rows after : {after:,}")
    print(f"  genes kept : {df['gene_norm'].nunique():,}")


df["support_type_norm"] = df[COLS["support_type"]].map(normalize_support_type)
df["miRNA_family_targetscan"] = df["miRNA_norm"].map(mirna_to_family)
df["miRNA_family"] = df["miRNA_family_targetscan"].fillna(
    df["miRNA_norm"].map(simple_mirna_family)
)

df["experiment_list"] = df[COLS["experiments"]].map(split_experiments)
df["experiment_classes"] = df["experiment_list"].map(
    lambda xs: sorted(set(classify_experiment(x) for x in xs)) if xs else []
)

if COLS["references"] is not None:
    df["study_ids"] = df[COLS["references"]].map(extract_pmids)
else:
    # if there is no explicit references column, fall back to miRTarBase ID as study proxy
    # this is weaker than PMID-based collapsing, but still usable
    fallback_col = COLS["mirtarbase_id"]
    if fallback_col is None:
        raise ValueError(
            "No references/PMID column detected, and no miRTarBase ID fallback found."
        )
    df["study_ids"] = df[fallback_col].astype(str).map(lambda x: [f"MIRTAR::{x}"])

# drop rows lacking key fields
df = df.dropna(subset=["miRNA_norm", "gene_norm", "support_type_norm"]).copy()

# optional: keep only recognized support types
# comment this out if you want to preserve unexpected values
df = df[df["support_type_norm"].isin(SUPPORT_TYPES)].copy()

print(f"\nNormalized rows retained: {len(df):,}")


# =============================================================================
# EXPLODE TO STUDY LEVEL
# =============================================================================

# one row can reference multiple PMIDs
df_study = df.explode("study_ids").rename(columns={"study_ids": "study_id"}).copy()
df_study["study_id"] = df_study["study_id"].astype(str).str.strip()
df_study = df_study[df_study["study_id"] != ""].copy()

print(f"Study-exploded rows: {len(df_study):,}")


# =============================================================================
# COLLAPSE TO (miRNA, gene, study) LEVEL
# =============================================================================

def collapse_interaction_study(group: pd.DataFrame) -> pd.Series:
    """
    One row per (miRNA, gene, study_id), preserving support-type presence and
    experiment-class presence, including experiment X support stratification.
    """
    out = {}

    out["miRNA"] = group["miRNA_norm"].iloc[0]
    out["miRNA_family"] = group["miRNA_family"].iloc[0]
    out["gene"] = group["gene_norm"].iloc[0]
    out["study_id"] = group["study_id"].iloc[0]

    # presence flags per support type
    support_set = set(group["support_type_norm"].dropna().tolist())
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        out[f"has_{st_slug}"] = int(st in support_set)

    # experiment class presence across the whole study for this interaction
    exp_classes = set()
    for xs in group["experiment_classes"]:
        exp_classes.update(xs)

    for ex in EXPERIMENT_CLASSES:
        out[f"has_{ex}"] = int(ex in exp_classes)

    # experiment x support presence
    for ex in EXPERIMENT_CLASSES:
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )

            mask = group["support_type_norm"].eq(st) & group["experiment_classes"].map(lambda xs: ex in xs)
            out[f"has_{ex}__{st_slug}"] = int(mask.any())

    return pd.Series(out)


interaction_study = (
    df_study
    .groupby(["miRNA_norm", "gene_norm", "study_id"], dropna=False)
    .apply(collapse_interaction_study)
    .reset_index(drop=True)
)

print(f"Collapsed interaction-study rows: {len(interaction_study):,}")


# =============================================================================
# TABLE 1: INTERACTION SUMMARY  (row = miRNA, gene)
# =============================================================================

def summarize_interaction(group: pd.DataFrame) -> pd.Series:
    out = {}

    out["miRNA"] = group["miRNA"].iloc[0]
    out["miRNA_family"] = group["miRNA_family"].iloc[0]
    out["gene"] = group["gene"].iloc[0]

    out["n_studies"] = group["study_id"].nunique()

    # support-type counts: all are publication counts
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        out[f"n_{st_slug}_studies"] = int(group[f"has_{st_slug}"].sum())

    # experiment-type counts: publication counts
    for ex in EXPERIMENT_CLASSES:
        out[f"n_{ex}_studies"] = int(group[f"has_{ex}"].sum())

    # experiment X support-type counts: publication counts
    exp_support_nested = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_nested[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            val = int(group[f"has_{ex}__{st_slug}"].sum())
            out[f"n_{ex}__{st_slug}_studies"] = val
            exp_support_nested[ex][st] = val

    out["experiment_support_counts_json"] = dict_to_json(exp_support_nested)

    # optional simple confidence score
    # tweak weights if you want
    out["evidence_score"] = (
        3 * int(group["has_reporter"].sum()) +
        3 * int(group["has_binding"].sum()) +
        2 * int(group["has_protein"].sum()) +
        1 * int(group["has_rna"].sum()) +
        1 * int(group["has_perturbation"].sum())
    )

    return pd.Series(out)


interaction_summary = (
    interaction_study
    .groupby(["miRNA", "gene"], dropna=False)
    .apply(summarize_interaction)
    .reset_index(drop=True)
)

print(f"Interaction summary rows: {len(interaction_summary):,}")


# =============================================================================
# TABLE 2: GENE SUMMARY  (row = gene)
# =============================================================================
def summarize_gene(group: pd.DataFrame) -> pd.Series:
    out = {}

    out["gene"] = group["gene"].iloc[0]

    out["n_unique_miRNAs"] = group["miRNA"].nunique()
    out["n_unique_families"] = group["miRNA_family"].nunique()
    out["n_total_studies"] = int(group["study_id"].nunique())

    # -------------------------------------------------------------------------
    # publication-count summaries
    # -------------------------------------------------------------------------
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        out[f"n_{st_slug}_studies"] = int(group[f"has_{st_slug}"].sum())

    for ex in EXPERIMENT_CLASSES:
        out[f"n_{ex}_studies"] = int(group[f"has_{ex}"].sum())

    exp_support_nested = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_nested[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            val = int(group[f"has_{ex}__{st_slug}"].sum())
            out[f"n_{ex}__{st_slug}_studies"] = val
            exp_support_nested[ex][st] = val
    out["experiment_support_counts_json"] = dict_to_json(exp_support_nested)

    # -------------------------------------------------------------------------
    # per-miRNA study count dicts
    # -------------------------------------------------------------------------
    mirna_study_counts = (
        group.groupby("miRNA")["study_id"]
        .nunique()
        .sort_values(ascending=False)
        .to_dict()
    )
    out["mirna_study_counts_json"] = dict_to_json(mirna_study_counts)

    family_study_counts = (
        group.groupby("miRNA_family")["study_id"]
        .nunique()
        .sort_values(ascending=False)
        .to_dict()
    )
    out["family_study_counts_json"] = dict_to_json(family_study_counts)

    # -------------------------------------------------------------------------
    # NEW: membership dicts (lists of miRNAs)
    # -------------------------------------------------------------------------

    # support type -> list of miRNAs
    support_to_mirnas = {}
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        mirs_in_bucket = sorted(
            group.loc[group[f"has_{st_slug}"] == 1, "miRNA"].dropna().unique().tolist()
        )
        support_to_mirnas[st] = mirs_in_bucket
    out["support_to_mirnas_json"] = dict_to_json(support_to_mirnas)

    # experiment type -> list of miRNAs
    experiment_to_mirnas = {}
    for ex in EXPERIMENT_CLASSES:
        mirs_in_bucket = sorted(
            group.loc[group[f"has_{ex}"] == 1, "miRNA"].dropna().unique().tolist()
        )
        experiment_to_mirnas[ex] = mirs_in_bucket
    out["experiment_to_mirnas_json"] = dict_to_json(experiment_to_mirnas)

    # experiment type x support type -> list of miRNAs
    exp_support_to_mirnas = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_to_mirnas[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            mirs_in_bucket = sorted(
                group.loc[group[f"has_{ex}__{st_slug}"] == 1, "miRNA"]
                .dropna()
                .unique()
                .tolist()
            )
            exp_support_to_mirnas[ex][st] = mirs_in_bucket
    out["experiment_support_to_mirnas_json"] = dict_to_json(exp_support_to_mirnas)

    # -------------------------------------------------------------------------
    # NEW: same idea but for families
    # -------------------------------------------------------------------------

    support_to_families = {}
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        fams_in_bucket = sorted(
            group.loc[group[f"has_{st_slug}"] == 1, "miRNA_family"]
            .dropna()
            .unique()
            .tolist()
        )
        support_to_families[st] = fams_in_bucket
    out["support_to_families_json"] = dict_to_json(support_to_families)

    experiment_to_families = {}
    for ex in EXPERIMENT_CLASSES:
        fams_in_bucket = sorted(
            group.loc[group[f"has_{ex}"] == 1, "miRNA_family"]
            .dropna()
            .unique()
            .tolist()
        )
        experiment_to_families[ex] = fams_in_bucket
    out["experiment_to_families_json"] = dict_to_json(experiment_to_families)

    exp_support_to_families = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_to_families[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            fams_in_bucket = sorted(
                group.loc[group[f"has_{ex}__{st_slug}"] == 1, "miRNA_family"]
                .dropna()
                .unique()
                .tolist()
            )
            exp_support_to_families[ex][st] = fams_in_bucket
    out["experiment_support_to_families_json"] = dict_to_json(exp_support_to_families)

    # -------------------------------------------------------------------------
    # detailed miRNA dict
    # -------------------------------------------------------------------------
    mirna_detail = {}
    for mir, sub in group.groupby("miRNA"):
        entry = {
            "n_studies": int(sub["study_id"].nunique()),
            "support_counts": {},
            "experiment_support_counts": {},
        }
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            entry["support_counts"][st] = int(sub[f"has_{st_slug}"].sum())

        for ex in EXPERIMENT_CLASSES:
            entry["experiment_support_counts"][ex] = {}
            for st in SUPPORT_TYPES:
                st_slug = (
                    st.lower()
                    .replace(" ", "_")
                    .replace("(", "")
                    .replace(")", "")
                    .replace("-", "")
                )
                entry["experiment_support_counts"][ex][st] = int(sub[f"has_{ex}__{st_slug}"].sum())

        mirna_detail[mir] = entry

    out["mirna_detail_json"] = dict_to_json(mirna_detail)

    return pd.Series(out)

gene_summary = (
    interaction_study
    .groupby("gene", dropna=False)
    .apply(summarize_gene)
    .reset_index(drop=True)
)

print(f"Gene summary rows: {len(gene_summary):,}")


# =============================================================================
# TABLE 3: miRNA SUMMARY  (row = miRNA)
# =============================================================================

def summarize_mirna(group: pd.DataFrame) -> pd.Series:
    out = {}

    out["miRNA"] = group["miRNA"].iloc[0]
    out["miRNA_family"] = group["miRNA_family"].iloc[0]

    out["n_unique_targets"] = group["gene"].nunique()
    out["n_total_studies"] = int(group["study_id"].nunique())
    out["studies_per_miRNA"] = int(group["study_id"].nunique())

    # -------------------------------------------------------------------------
    # publication-count summaries
    # -------------------------------------------------------------------------
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        out[f"n_{st_slug}_studies"] = int(group[f"has_{st_slug}"].sum())

    for ex in EXPERIMENT_CLASSES:
        out[f"n_{ex}_studies"] = int(group[f"has_{ex}"].sum())

    exp_support_nested = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_nested[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            val = int(group[f"has_{ex}__{st_slug}"].sum())
            out[f"n_{ex}__{st_slug}_studies"] = val
            exp_support_nested[ex][st] = val
    out["experiment_support_counts_json"] = dict_to_json(exp_support_nested)

    # -------------------------------------------------------------------------
    # per-gene study counts
    # -------------------------------------------------------------------------
    gene_study_counts = (
        group.groupby("gene")["study_id"]
        .nunique()
        .sort_values(ascending=False)
        .to_dict()
    )
    out["target_gene_study_counts_json"] = dict_to_json(gene_study_counts)

    # -------------------------------------------------------------------------
    # NEW: membership dicts (lists of genes)
    # -------------------------------------------------------------------------

    # support type -> list of genes
    support_to_genes = {}
    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        genes_in_bucket = sorted(
            group.loc[group[f"has_{st_slug}"] == 1, "gene"].dropna().unique().tolist()
        )
        support_to_genes[st] = genes_in_bucket
    out["support_to_genes_json"] = dict_to_json(support_to_genes)

    # experiment type -> list of genes
    experiment_to_genes = {}
    for ex in EXPERIMENT_CLASSES:
        genes_in_bucket = sorted(
            group.loc[group[f"has_{ex}"] == 1, "gene"].dropna().unique().tolist()
        )
        experiment_to_genes[ex] = genes_in_bucket
    out["experiment_to_genes_json"] = dict_to_json(experiment_to_genes)

    # experiment type x support type -> list of genes
    exp_support_to_genes = {}
    for ex in EXPERIMENT_CLASSES:
        exp_support_to_genes[ex] = {}
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            genes_in_bucket = sorted(
                group.loc[group[f"has_{ex}__{st_slug}"] == 1, "gene"]
                .dropna()
                .unique()
                .tolist()
            )
            exp_support_to_genes[ex][st] = genes_in_bucket
    out["experiment_support_to_genes_json"] = dict_to_json(exp_support_to_genes)

    # -------------------------------------------------------------------------
    # detailed target dict
    # -------------------------------------------------------------------------
    target_detail = {}
    for gene, sub in group.groupby("gene"):
        entry = {
            "n_studies": int(sub["study_id"].nunique()),
            "support_counts": {},
            "experiment_support_counts": {},
        }
        for st in SUPPORT_TYPES:
            st_slug = (
                st.lower()
                .replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("-", "")
            )
            entry["support_counts"][st] = int(sub[f"has_{st_slug}"].sum())

        for ex in EXPERIMENT_CLASSES:
            entry["experiment_support_counts"][ex] = {}
            for st in SUPPORT_TYPES:
                st_slug = (
                    st.lower()
                    .replace(" ", "_")
                    .replace("(", "")
                    .replace(")", "")
                    .replace("-", "")
                )
                entry["experiment_support_counts"][ex][st] = int(sub[f"has_{ex}__{st_slug}"].sum())

        target_detail[gene] = entry

    out["target_gene_detail_json"] = dict_to_json(target_detail)

    return pd.Series(out)

mirna_summary = (
    interaction_study
    .groupby("miRNA", dropna=False)
    .apply(summarize_mirna)
    .reset_index(drop=True)
)

print(f"miRNA summary rows: {len(mirna_summary):,}")


# =============================================================================
# OPTIONAL: A FAMILY SUMMARY TABLE
# =============================================================================

def summarize_family(group: pd.DataFrame) -> pd.Series:
    out = {}

    out["miRNA_family"] = group["miRNA_family"].iloc[0]
    out["n_unique_miRNAs"] = group["miRNA"].nunique()
    out["n_unique_targets"] = group["gene"].nunique()
    out["n_total_studies"] = int(group["study_id"].nunique())

    for st in SUPPORT_TYPES:
        st_slug = (
            st.lower()
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("-", "")
        )
        out[f"n_{st_slug}_studies"] = int(group[f"has_{st_slug}"].sum())

    for ex in EXPERIMENT_CLASSES:
        out[f"n_{ex}_studies"] = int(group[f"has_{ex}"].sum())

    return pd.Series(out)


family_summary = (
    interaction_study
    .groupby("miRNA_family", dropna=False)
    .apply(summarize_family)
    .reset_index(drop=True)
)

print(f"Family summary rows: {len(family_summary):,}")


# =============================================================================
# SAVE
# =============================================================================

out_dir = "data/miRNA/test"
os.makedirs(out_dir, exist_ok=True)
interaction_study.to_csv(f"{out_dir}/mirtar_interaction_study_collapsed.csv", index=False)
interaction_summary.to_csv(f"{out_dir}/mirtar_interaction_summary.csv", index=False)
gene_summary.to_csv(f"{out_dir}/mirtar_gene_summary.csv", index=False)
mirna_summary.to_csv(f"{out_dir}/mirtar_mirna_summary.csv", index=False)
family_summary.to_csv(f"{out_dir}/mirtar_family_summary_simple.csv", index=False)

print("\nSaved:")
print(f"  {out_dir}/mirtar_interaction_study_collapsed.csv")
print(f"  {out_dir}/mirtar_interaction_summary.csv")
print(f"  {out_dir}/mirtar_gene_summary.csv")
print(f"  {out_dir}/mirtar_mirna_summary.csv")
print(f"  {out_dir}/mirtar_family_summary_simple.csv")

Detected columns:
  mirtarbase_id: miRTarBase ID
  mirna: miRNA
  mirna_species: Species (miRNA)
  gene: Target Gene
  gene_entrez: Target Gene (Entrez ID)
  gene_species: Species (Target Gene)
  experiments: Experiments
  support_type: Support Type
  references: None
Filtered to PRIMARY_GENES:
  rows before: 4,004,967
  rows after : 19,517
  genes kept : 63

Normalized rows retained: 19,517
Study-exploded rows: 19,517


/tmp/ipykernel_36923/1709110617.py:404: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_interaction_study)


Collapsed interaction-study rows: 8,257


/tmp/ipykernel_36923/1709110617.py:473: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_interaction)


Interaction summary rows: 8,250


...

Gene summary rows: 63


/tmp/ipykernel_36923/1709110617.py:842: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_mirna)


miRNA summary rows: 2,079


/tmp/ipykernel_36923/1709110617.py:880: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_family)


Family summary rows: 1,701

Saved:
  data/miRNA/test/mirtar_interaction_study_collapsed.csv
  data/miRNA/test/mirtar_interaction_summary.csv
  data/miRNA/test/mirtar_gene_summary.csv
  data/miRNA/test/mirtar_mirna_summary.csv
  data/miRNA/test/mirtar_family_summary_simple.csv


In [81]:
gene_summary

,gene,n_unique_miRNAs,n_unique_families,n_total_studies,n_functional_mti_studies,n_functional_mti_weak_studies,n_nonfunctional_mti_studies,n_nonfunctional_mti_weak_studies,n_reporter_studies,n_binding_studies,...,experiment_support_counts_json,mirna_study_counts_json,family_study_counts_json,support_to_mirnas_json,experiment_to_mirnas_json,experiment_support_to_mirnas_json,support_to_families_json,experiment_to_families_json,experiment_support_to_families_json,mirna_detail_json
0,ADAM10,171,150,171,5,166,1,0,4,164,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-1"": 1, ""hsa-miR-103a"": 1, ""hsa-miR-1...","{""miR-1"": 1, ""miR-1-3p/206"": 2, ""miR-103-3p/10...","{""Functional MTI"": [""hsa-miR-122-5p"", ""hsa-miR...","{""binding"": [""hsa-miR-1"", ""hsa-miR-103a"", ""hsa...","{""binding"": {""Functional MTI"": [], ""Functional...","{""Functional MTI"": [""miR-122-5p"", ""miR-140"", ""...","{""binding"": [""miR-1"", ""miR-1-3p/206"", ""miR-103...","{""binding"": {""Functional MTI"": [], ""Functional...","{""hsa-miR-1"": {""experiment_support_counts"": {""..."
1,ADAM17,153,112,153,6,151,4,1,6,149,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-105"": 1, ""hsa-miR-1208"": 1, ""hsa-miR...","{""miR-105"": 1, ""miR-1208"": 1, ""miR-122"": 1, ""m...","{""Functional MTI"": [""hsa-miR-122-5p"", ""hsa-miR...","{""binding"": [""hsa-miR-105"", ""hsa-miR-1208"", ""h...","{""binding"": {""Functional MTI"": [], ""Functional...","{""Functional MTI"": [""miR-122-5p"", ""miR-124"", ""...","{""binding"": [""miR-105"", ""miR-1208"", ""miR-122"",...","{""binding"": {""Functional MTI"": [], ""Functional...","{""hsa-miR-105"": {""experiment_support_counts"": ..."
2,B2M,113,99,113,0,113,0,0,0,113,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-105"": 1, ""hsa-miR-106a"": 1, ""hsa-miR...","{""miR-105"": 1, ""miR-106a"": 1, ""miR-106b"": 1, ""...","{""Functional MTI"": [], ""Functional MTI (Weak)""...","{""binding"": [""hsa-miR-105"", ""hsa-miR-106a"", ""h...","{""binding"": {""Functional MTI"": [], ""Functional...","{""Functional MTI"": [], ""Functional MTI (Weak)""...","{""binding"": [""miR-105"", ""miR-106a"", ""miR-106b""...","{""binding"": {""Functional MTI"": [], ""Functional...","{""hsa-miR-105"": {""experiment_support_counts"": ..."
3,CALR,209,173,209,0,209,0,0,0,205,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-1-3p"": 1, ""hsa-miR-1207-5p"": 1, ""hsa...","{""miR-1-3p/206"": 1, ""miR-1207-5p/4763-3p"": 2, ...","{""Functional MTI"": [], ""Functional MTI (Weak)""...","{""binding"": [""hsa-miR-1207-5p"", ""hsa-miR-1225-...","{""binding"": {""Functional MTI"": [], ""Functional...","{""Functional MTI"": [], ""Functional MTI (Weak)""...","{""binding"": [""miR-1207-5p/4763-3p"", ""miR-1225-...","{""binding"": {""Functional MTI"": [], ""Functional...","{""hsa-miR-1-3p"": {""experiment_support_counts"":..."
4,CCL5,155,120,155,5,150,0,1,5,146,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-106a"": 1, ""hsa-miR-106a-5p"": 1, ""hsa...","{""let-7-5p/98-5p"": 1, ""miR-106a"": 1, ""miR-106b...","{""Functional MTI"": [""hsa-miR-146a-5p"", ""hsa-mi...","{""binding"": [""hsa-miR-106a"", ""hsa-miR-106a-5p""...","{""binding"": {""Functional MTI"": [], ""Functional...","{""Functional MTI"": [""let-7-5p/98-5p"", ""miR-146...","{""binding"": [""miR-106a"", ""miR-106b"", ""miR-1184...","{""binding"": {""Functional MTI"": [], ""Functional...","{""hsa-miR-106a"": {""experiment_support_counts"":..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,TGFB1,46,41,47,20,26,2,2,15,17,...,"{""binding"": {""Functional MTI"": 0, ""Functional ...","{""hsa-miR-106a-3p"": 1, ""hsa-miR-122-5p"": 1, ""h...","{""miR-106a-3p"": 1, ""miR-122-5p"": 1, ""miR-1237-...","{""Functional MTI"": [""hsa-miR-106a-3p"", ""hsa-mi...","{""binding"": [""hsa-miR-1908"", ""hsa-miR-3180"", ""...","{""binding"": {""Functional MTI"": [], "

In [71]:
target_info = pd.read_csv("data/miRNA/miR_Family_Info.txt", sep="\t")
target_info.head()



,miR family,Seed+m8,Species ID,MiRBase ID,Mature sequence,Family Conservation?,MiRBase Accession
0,let-7,UAUACGA,9615,cfa-let-7d,CUAUACGACCUGCUGCCUUUCUUAG,-1,MIMAT0034396
1,let-7/98,GAGGUAG,8364,xtr-let-7a,UGAGGUAGUAGGUUGUAUAGUU,2,MIMAT0003667
2,let-7/98,GAGGUAG,8364,xtr-let-7b,UGAGGUAGUAGUUUGUGUAGUU,2,MIMAT0003550
3,let-7/98,GAGGUAG,8364,xtr-let-7c,UGAGGUAGUAGGUUGUAUGGUU,2,MIMAT0003644
4,let-7/98,GAGGUAG,8364,xtr-let-7e,UGAGGUAGUAGGUUGUUUAGUU,2,MIMAT0003666


In [73]:
target_info["Species ID"].value_counts()

Species ID
9606     2606
10090    1937
9031      994
9544      914
9913      793
13616     767
10116     765
9598      587
9615      453
8364      178
Name: count, dtype: int64

In [78]:
target_info[target_info["MiRBase ID"] == "hsa-miR-122-5p"]

,miR family,Seed+m8,Species ID,MiRBase ID,Mature sequence,Family Conservation?,MiRBase Accession
399,miR-122-5p,GGAGUGU,9606,hsa-miR-122-5p,UGGAGUGUGACAAUGGUGUUUG,2,MIMAT0000421


In [77]:
mirs[mirs["miRNA"].isin(target_info["MiRBase ID"].unique().tolist())]

,miRTarBase ID,miRNA,Species (miRNA),Target Gene,Target Gene (Entrez ID),Species (Target Gene),Experiments,Support Type,References (PMID)
0,MIRT003105,hsa-miR-122-5p,hsa,SLC7A1,6541.0,hsa,Luciferase reporter assay//Western blot,Functional MTI,17179747.0
1,MIRT003105,hsa-miR-122-5p,hsa,SLC7A1,6541.0,hsa,Luciferase reporter assay//Western blot,Non-Functional MTI,17179747.0
2,MIRT003105,hsa-miR-122-5p,hsa,SLC7A1,6541.0,hsa,Luciferase reporter assay//Western blot,Non-Functional MTI,17179747.0
3,MIRT003105,hsa-miR-122-5p,hsa,SLC7A1,6541.0,hsa,Luciferase reporter assay//Western blot,Functional MTI,17179747.0
4,MIRT003105,hsa-miR-122-5p,hsa,SLC7A1,6541.0,hsa,Luciferase reporter assay//Western blot,Functional MTI,17179747.0
...,...,...,...,...,...,...,...,...,...
4004959,MIRT2164641,hsa-miR-512-3p,hsa,ZZZ3,26009.0,hsa,CLIP-seq,Functional MTI (Weak),NaN
4004961,MIRT2383704,hsa-miR-543,hsa,ZZZ3,26009.0,hsa,CLIP-seq,Functional MTI (Weak),NaN
4004963,MIRT2383709,hsa-miR-556-5p,hsa,ZZZ3,26009.0,hsa,CLIP-seq,Functional MTI (Weak),NaN
4004964,MIRT2383710,hsa-miR-558,hsa,ZZZ3,26009.0,hsa,CLIP-seq,Functional MTI (Weak),NaN
